In [2]:
# Setup the Jupyter version of Dash
from jupyter_dash import JupyterDash

# Configure the necessary Python module imports for dashboard componen# Setup the Jupyter version of Dash
from jupyter_dash import JupyterDash

# Configure the necessary Python module imports for dashboard components
import dash_leaflet as dl
from dash import dcc, html
import plotly.express as px
from dash import dash_table
from dash.dependencies import Input, Output, State
import base64
JupyterDash.infer_jupyter_proxy_config()

# Configure OS routines
import os

# Configure the plotting routines
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


#### FIX ME #####
# change animal_shelter and AnimalShelter to match your CRUD Python module file name and class name
from CRUD_Python_Module import AnimalShelter

###########################
# Data Manipulation / Model
###########################
# FIX ME update with your username and password and CRUD Python module name

username = "aacuser"
password = "Brandon1994!"

# Connect to database via CRUD Module
db = AnimalShelter(username, password)

# class read method must support return of list object and accept projection json input
# sending the read method an empty document requests all documents be returned
df = pd.DataFrame.from_records(db.read({}))

# MongoDB v5+ is going to return the '_id' column and that is going to have an 
# invlaid object type of 'ObjectID' - which will cause the data_table to crash - so we remove
# it in the dataframe here. The df.drop command allows us to drop the column. If we do not set
# inplace=True - it will reeturn a new dataframe that does not contain the dropped column(s)
if '_id' in df.columns:
    df.drop(columns=['_id'], inplace=True)

## Debug
# print(len(df.to_dict(orient='records')))
# print(df.columns)

# convert map coordinates to numeric values
if 'location_lat' in df.columns:
    df['location_lat'] = pd.to_numeric(df['location_lat'], errors='coerce')
if 'location_long' in df.columns:
    df['location_long'] = pd.to_numeric(df['location_long'], errors='coerce')


#########################
# Dashboard Layout / View
#########################
app = JupyterDash(__name__)

#FIX ME Add in Grazioso Salvare’s logo
# replace this with the exact filename/path in your Codio project if needed
image_filename = 'Grazioso Salvare Logo.png'
encoded_image = base64.b64encode(open(image_filename, 'rb').read())

#FIX ME Place the HTML image tag in the line below into the app.layout code according to your design
#FIX ME Also remember to include a unique identifier such as your name or date
#html.Img(src='data:image/png;base64,{}'.format(encoded_image.decode()))

app.layout = html.Div([
#    html.Div(id='hidden-div', style={'display':'none'}),
    html.Center(html.B(html.H1('CS-340 Dashboard'))),
    html.Hr(),

    html.Div([
        html.Center(
            html.A(
                html.Img(
                    src='data:image/png;base64,{}'.format(encoded_image.decode()),
                    style={'height': '200px'}
                ),
                href='https://www.snhu.edu',
                target='_blank'
            )
        ),
        html.Center(html.H3("Brandon Burks"))
    ]),

    html.Hr(),
    html.Div(
        
#FIXME Add in code for the interactive filtering options. For example, Radio buttons, drop down, checkboxes, etc.
        [
            dcc.RadioItems(
                id='filter-type',
                options=[
                    {'label': ' Water Rescue', 'value': 'Water Rescue'},
                    {'label': ' Mountain or Wilderness Rescue', 'value': 'Mountain or Wilderness Rescue'},
                    {'label': ' Disaster or Individual Tracking', 'value': 'Disaster or Individual Tracking'},
                    {'label': ' Reset', 'value': 'Reset'}
                ],
                value='Reset',
                inline=True
            )
        ]
    ),
    html.Hr(),
    dash_table.DataTable(id='datatable-id',
                         columns=[{"name": i, "id": i, "deletable": False, "selectable": True} for i in df.columns],
                         data=df.to_dict('records'),
#FIXME: Set up the features for your interactive data table to make it user-friendly for your client
#If you completed the Module Six Assignment, you can copy in the code you created here 
                         page_size=10,
                         sort_action='native',
                         filter_action='native',
                         row_selectable='single',
                         selected_rows=[0],
                         style_table={'overflowX': 'auto'},
                         style_cell={
                             'minWidth': '120px',
                             'width': '120px',
                             'maxWidth': '180px',
                             'whiteSpace': 'normal',
                             'textAlign': 'left'
                         },

                        ),
    html.Br(),
    html.Hr(),
#This sets up the dashboard so that your chart and your geolocation chart are side-by-side
    html.Div(className='row',
         style={'display' : 'flex'},
             children=[
        html.Div(
            id='graph-id',
            className='col s12 m6',

            ),
        html.Div(
            id='map-id',
            className='col s12 m6',
            )
        ])
])

#############################################
# Interaction Between Components / Controller
#############################################

def get_filter_query(filter_type):
    if filter_type == "Water Rescue":
        return {
            "animal_type": "Dog",
            "sex_upon_outcome": {"$regex": "Intact Female", "$options": "i"},
            "age_upon_outcome_in_weeks": {"$gte": 26, "$lte": 156},
            "$or": [
                {"breed": {"$regex": "Labrador Retriever Mix", "$options": "i"}},
                {"breed": {"$regex": "Chesapeake Bay Retriever", "$options": "i"}},
                {"breed": {"$regex": "Newfoundland", "$options": "i"}}
            ]
        }

    elif filter_type == "Mountain or Wilderness Rescue":
        return {
            "animal_type": "Dog",
            "sex_upon_outcome": {"$regex": "Intact Male", "$options": "i"},
            "age_upon_outcome_in_weeks": {"$gte": 26, "$lte": 156},
            "$or": [
                {"breed": {"$regex": "German Shepherd", "$options": "i"}},
                {"breed": {"$regex": "Alaskan Malamute", "$options": "i"}},
                {"breed": {"$regex": "Old English Sheepdog", "$options": "i"}},
                {"breed": {"$regex": "Siberian Husky", "$options": "i"}},
                {"breed": {"$regex": "Rottweiler", "$options": "i"}}
            ]
        }

    elif filter_type == "Disaster or Individual Tracking":
        return {
            "animal_type": "Dog",
            "sex_upon_outcome": {"$regex": "Intact Male", "$options": "i"},
            "age_upon_outcome_in_weeks": {"$gte": 20, "$lte": 300},
            "$or": [
                {"breed": {"$regex": "Doberman Pinscher", "$options": "i"}},
                {"breed": {"$regex": "German Shepherd", "$options": "i"}},
                {"breed": {"$regex": "Golden Retriever", "$options": "i"}},
                {"breed": {"$regex": "Bloodhound", "$options": "i"}},
                {"breed": {"$regex": "Rottweiler", "$options": "i"}}
            ]
        }

    else:
        return {}


    
@app.callback(Output('datatable-id','data'),
              [Input('filter-type', 'value')])
def update_dashboard(filter_type):
## FIX ME Add code to filter interactive data table with MongoDB queries
#
#        
#        columns=[{"name": i, "id": i, "deletable": False, "selectable": True} for i in df.columns]
#        data=df.to_dict('records')
#       
#       
#        return (data,columns)

    query = get_filter_query(filter_type)
    filtered_df = pd.DataFrame.from_records(db.read(query))

    if '_id' in filtered_df.columns:
        filtered_df.drop(columns=['_id'], inplace=True)

    if 'location_lat' in filtered_df.columns:
        filtered_df['location_lat'] = pd.to_numeric(filtered_df['location_lat'], errors='coerce')
    if 'location_long' in filtered_df.columns:
        filtered_df['location_long'] = pd.to_numeric(filtered_df['location_long'], errors='coerce')

    return filtered_df.to_dict('records')

# Display the breeds of animal based on quantity represented in
# the data table
@app.callback(
    Output('graph-id', "children"),
    [Input('datatable-id', "derived_virtual_data")])
def update_graphs(viewData):
    ###FIX ME ####
    # add code for chart of your choice (e.g. pie chart) #

    if viewData is None:
        dff = df.copy()
    else:
        dff = pd.DataFrame.from_dict(viewData)

    if dff.empty or 'breed' not in dff.columns:
        return [
            dcc.Graph(
                figure=px.pie(title='Preferred Animals')
            )
        ]

    breed_counts = dff['breed'].value_counts().reset_index()
    breed_counts.columns = ['breed', 'count']

    return [
        dcc.Graph(            
            figure = px.pie(breed_counts, names='breed', values='count', title='Preferred Animals')
        )    
    ]
    
#This callback will highlight a cell on the data table when the user selects it
@app.callback(
    Output('datatable-id', 'style_data_conditional'),
    Input('datatable-id', 'selected_columns')
)
def update_styles(selected_columns):
    if selected_columns is None:
        return []

    return [{
        'if': {'column_id': i},
        'background_color': '#D2F3FF'
    } for i in selected_columns]


# This callback will update the geo-location chart for the selected data entry
# derived_virtual_data will be the set of data available from the datatable in the form of 
# a dictionary.
# derived_virtual_selected_rows will be the selected row(s) in the table in the form of
# a list. For this application, we are only permitting single row selection so there is only
# one value in the list.
# The iloc method allows for a row, column notation to pull data from the datatable
@app.callback(
    Output('map-id', "children"),
    [Input('datatable-id', "derived_virtual_data"),
     Input('datatable-id', "derived_virtual_selected_rows")])
def update_map(viewData, index):  
    if viewData is None:
        dff = df.copy()
    else:
        dff = pd.DataFrame.from_dict(viewData)

    if dff.empty:
        return [html.Div("No data available for map display.")]

    if 'location_lat' not in dff.columns or 'location_long' not in dff.columns:
        return [html.Div("Location data not available.")]

    dff['location_lat'] = pd.to_numeric(dff['location_lat'], errors='coerce')
    dff['location_long'] = pd.to_numeric(dff['location_long'], errors='coerce')
    dff = dff.dropna(subset=['location_lat', 'location_long'])

    if dff.empty:
        return [html.Div("No valid coordinates available for map display.")]

    # Because we only allow single row selection, the list can be converted to a row index here
    if index is None or len(index) == 0:
        row = 0
    else: 
        row = index[0]

    if row >= len(dff):
        row = 0
        
    # Austin TX is at [30.75,-97.48]
    return [
        dl.Map(style={'width': '1000px', 'height': '500px'}, center=[30.75,-97.48], zoom=10, children=[
            dl.TileLayer(id="base-layer-id"),
            # Marker with tool tip and popup
            # Column 13 and 14 define the grid-coordinates for the map
            # Column 4 defines the breed for the animal
            # Column 9 defines the name of the animal
            dl.Marker(position=[dff.iloc[row]['location_lat'], dff.iloc[row]['location_long']], children=[
                dl.Tooltip(str(dff.iloc[row]['breed'])),
                dl.Popup([
                    html.H1("Animal Name"),
                    html.P(str(dff.iloc[row]['name']))
                ])
            ])
        ])
    ]


# Run app and display result in jupyterlab mode, note, if you have previously run a prior app, the default port of 8050 may not be available, if so, try setting an alternate port.
app.run_server(mode='inline')